# ICM-UADE — Mapa de precios por sucursal
**INECO · Universidad Argentina de la Empresa**

Genera el mapa interactivo, rankings de cadenas, provincias y barrios CABA.

▶️ **Ejecutar todo** (`Runtime → Run all`) y esperar ~10 minutos.
La única celda modificable es la primera — para elegir el mes.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CONFIGURACIÓN — única celda que podés modificar
# ══════════════════════════════════════════════════════════════════

# Mes a analizar (formato 'YYYY-MM').
# Dejalo en None para usar el último mes disponible automáticamente.
MES = None   # ← cambiá a '2026-04' si querés un mes específico

# ── No modificar nada de acá para abajo ───────────────────────────
REPO_URL     = 'https://github.com/santiagoriverti/analisis_precios_minoristas_UADE.git'
MANIFEST_URL = 'https://raw.githubusercontent.com/santiagoriverti/analisis_precios_minoristas_UADE/main/data/drive_manifest.json'
DIR_DATOS    = '/content/datos_sepa'
DIR_REPO     = '/content/repo'

# IDs de las carpetas de Drive de INECO (no modificar)
FOLDER_HISTORICO = '13GONeBs5lQCSUdBioHYk-8GhfDtIyliD'  # 2018-2023
FOLDER_RECIENTE  = '1GNs9SrZ4BIoBsviBVWYYqRcsj4dwPF-I'  # 2024-presente

NOMBRE_MAESTRO_PROD = 'Maestro de Productos Interno.xlsx'
NOMBRE_MAESTRO_SUC  = 'maestro_sucursales_completo.xlsx'

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 1 — Instalación y clonado
# ══════════════════════════════════════════════════════════════════
import os, sys

if not os.path.exists(DIR_REPO):
    os.system(f'git clone -q {REPO_URL} {DIR_REPO}')
    print('✓ Repositorio clonado')
else:
    os.system(f'git -C {DIR_REPO} pull -q')
    print('✓ Repositorio actualizado')

os.system('pip install -q gdown folium branca openpyxl pyarrow')
sys.path.insert(0, f'{DIR_REPO}/src')
os.makedirs(DIR_DATOS, exist_ok=True)
print('✓ Listo')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 2 — Conectar a Google Drive y preparar la descarga
# ══════════════════════════════════════════════════════════════════
import re
from datetime import date
from google.colab import auth
from googleapiclient.discovery import build

# ── Autenticar (una sola vez por sesión) ─────────────────────────
print('Conectando a Google Drive...')
auth.authenticate_user()
_drive = build('drive', 'v3', cache_discovery=False)
print('✓ Conectado')

# ── Listar todos los archivos en ambas carpetas ───────────────────
def _listar_folder(svc, folder_id):
    items, tok = {}, None
    while True:
        r = svc.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields='nextPageToken,files(id,name,size)',
            pageSize=1000, pageToken=tok
        ).execute()
        for f in r.get('files', []):
            items[f['name']] = f
        tok = r.get('nextPageToken')
        if not tok:
            break
    return items

print('Listando archivos en Drive...')
DRIVE_FILES = {}
for _fid in (FOLDER_HISTORICO, FOLDER_RECIENTE):
    DRIVE_FILES.update(_listar_folder(_drive, _fid))

_zips_datos = sorted(
    n for n in DRIVE_FILES
    if n.lower().endswith('.zip') and 'apoyo' not in n.lower()
)
print(f'  Archivos en Drive   : {len(DRIVE_FILES)}')
print(f'  ZIPs de datos SEPA  : {_zips_datos}')

# ── Determinar mes a analizar ─────────────────────────────────────
if MES:
    MES_ANALIZAR = MES
else:
    _hoy = date.today()
    _m = _hoy.month - 1 if _hoy.month > 1 else 12
    _a = _hoy.year if _hoy.month > 1 else _hoy.year - 1
    MES_ANALIZAR = f'{_a}-{_m:02d}'
    print(f'\n  MES no especificado → último mes completo estimado: {MES_ANALIZAR}')

_year  = int(MES_ANALIZAR[:4])
_month = int(MES_ANALIZAR[5:7])
MES_MMAAAA = f'{_month:02d}{_year}'   # ej: '042026' para 2026-04

# ── Elegir ZIP a descargar ────────────────────────────────────────
# Prioridad: ZIP semestral (2021A, 2021B) > ZIP anual (2026.zip)
_sem = 'A' if _month <= 6 else 'B'
ZIP_TARGET = next(
    (c for c in [f'{_year}{_sem}.zip', f'{_year}.zip'] if c in DRIVE_FILES),
    None
)

if ZIP_TARGET is None:
    raise RuntimeError(
        f'\nNo se encontró ZIP para {MES_ANALIZAR} en Drive.\n'
        f'  Buscaba : {_year}{_sem}.zip  o  {_year}.zip\n'
        f'  ZIPs disponibles: {_zips_datos}\n'
        f'  Revisá que los datos de {_year} estén subidos a Drive.'
    )

_size_gb = int(DRIVE_FILES[ZIP_TARGET].get('size', 0)) / 1e9
print(f'\n  Mes a analizar : {MES_ANALIZAR}')
print(f'  ZIP a descargar: {ZIP_TARGET}  ({_size_gb:.2f} GB)')
print('\n✓ PASO 2 listo — ejecutá el siguiente paso para descargar')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 3 — Descargar ZIP y extraer archivos del mes
# ══════════════════════════════════════════════════════════════════
import re, zipfile, shutil
from pathlib import Path
from googleapiclient.http import MediaIoBaseDownload

# ── Función de descarga robusta ───────────────────────────────────
def _download(svc, file_id, dest, label, expected_bytes=0):
    """Descarga un archivo de Drive. Verifica integridad por tamaño."""
    dest = Path(dest)
    if dest.exists():
        actual = dest.stat().st_size
        if expected_bytes > 0 and actual == expected_bytes:
            print(f'  ↩  {label}  ({actual/1e6:.0f} MB — ya completo)')
            return dest
        elif expected_bytes == 0 and actual > 50_000:
            print(f'  ↩  {label}  ({actual/1e6:.0f} MB)')
            return dest
        elif expected_bytes > 0 and actual != expected_bytes:
            print(f'  ⚠  {label} incompleto ({actual/1e6:.0f}/{expected_bytes/1e6:.0f} MB) — re-descargando')
    print(f'  ↓  Descargando {label}...')
    req = svc.files().get_media(fileId=file_id)
    with open(dest, 'wb') as fh:
        dl = MediaIoBaseDownload(fh, req, chunksize=100 * 1024 * 1024)
        done = False
        while not done:
            status, done = dl.next_chunk()
            if status:
                try:
                    pct = int(status.progress() * 100)
                    mb  = status.resumable_progress / 1e6
                    print(f'\r     {pct:>3}%  ({mb:.0f} MB)', end='', flush=True)
                except Exception:
                    pass
    print(f'\r  ✓  {label}  ({dest.stat().st_size/1e6:.0f} MB)                        ')
    return dest

# ── 1. Descargar ZIP anual/semestral ─────────────────────────────
_zip_info  = DRIVE_FILES[ZIP_TARGET]
_zip_local = _download(
    _drive, _zip_info['id'],
    Path(DIR_DATOS) / ZIP_TARGET,
    ZIP_TARGET,
    int(_zip_info.get('size', 0))
)

# ── 2. Limpiar CSV.GZ de otros meses (evita confusión al cargar) ─
for _stale in sorted(Path(DIR_DATOS).glob('*.csv.gz')):
    if MES_MMAAAA not in _stale.name:
        print(f'  ♻  Eliminando CSV.GZ de otro mes: {_stale.name}')
        _stale.unlink()

# ── 3. Inspeccionar ZIP y extraer CSV.GZ del mes target ───────────
print(f'\n  Inspeccionando {ZIP_TARGET}...')
with zipfile.ZipFile(_zip_local, 'r') as _z:
    _todo  = _z.namelist()
    _csvgz = [n for n in _todo if n.lower().endswith('.csv.gz')]
    print(f'     Total entradas en ZIP : {len(_todo)}')
    print(f'     Archivos .csv.gz      : {len(_csvgz)}')
    if _csvgz:
        print(f'     Muestra              : {[Path(n).name for n in sorted(_csvgz)[:6]]}')

    # Patrón flexible: MMAAAA_pais_parte[12]*(COMPLETO)?.csv.gz
    # Acepta:  042026_pais_parte1COMPLETO.csv.gz
    #          042026_pais_parte1_COMPLETO.csv.gz
    #          042026_pais_parte2COMPLETO.csv.gz   etc.
    _pat = re.compile(
        rf'^{re.escape(MES_MMAAAA)}_pais_parte[12].*\.csv\.gz$',
        re.IGNORECASE
    )
    _mes_files = [n for n in _csvgz if _pat.match(Path(n).name)]

    if len(_mes_files) < 2:
        _meses_zip = sorted({
            f"{_m.group(2)}-{_m.group(1)}"
            for n in _csvgz
            if (_m := re.search(r'(\d{2})(\d{4})_pais_parte', Path(n).name))
        })
        raise RuntimeError(
            f'\nNo se encontraron los 2 archivos para {MES_ANALIZAR} en {ZIP_TARGET}.\n'
            f'  Encontrados: {[Path(n).name for n in _mes_files]}\n'
            f'  Todos los CSV.GZ: {[Path(n).name for n in sorted(_csvgz)]}\n'
            f'  Meses detectados en ZIP: {_meses_zip}\n'
            f'  → Si tu mes figura arriba, cambiá MES en la celda de configuración.'
        )

    print(f'     Del mes {MES_ANALIZAR}: {[Path(n).name for n in sorted(_mes_files)]}')

    for _src in sorted(_mes_files):
        _nombre = Path(_src).name
        _dest   = Path(DIR_DATOS) / _nombre
        if _dest.exists() and _dest.stat().st_size > 50_000:
            print(f'     ↩  {_nombre}  ({_dest.stat().st_size/1e6:.0f} MB — ya extraído)')
            continue
        print(f'     Extrayendo {_nombre}...', end='', flush=True)
        _dest.write_bytes(_z.read(_src))
        print(f'  ✓  ({_dest.stat().st_size/1e6:.0f} MB)')

# ── 4. Maestros (Archivos_de_apoyo.zip) ───────────────────────────
print()
_faltantes = [
    n for n in (NOMBRE_MAESTRO_PROD, NOMBRE_MAESTRO_SUC)
    if not (Path(DIR_DATOS) / n).exists()
    or (Path(DIR_DATOS) / n).stat().st_size < 50_000
]

if _faltantes:
    _APOYO = 'Archivos_de_apoyo.zip'
    if _APOYO not in DRIVE_FILES:
        raise RuntimeError(
            f'\nFaltan los archivos maestros y no existe "{_APOYO}" en Drive.\n'
            f'  Faltantes: {_faltantes}\n'
            f'  Solución: subí los maestros a Drive dentro de un ZIP llamado "{_APOYO}".'
        )
    _apoyo_local = _download(
        _drive, DRIVE_FILES[_APOYO]['id'],
        Path(DIR_DATOS) / _APOYO,
        _APOYO,
        int(DRIVE_FILES[_APOYO].get('size', 0))
    )
    print(f'  Extrayendo maestros de {_APOYO}...')
    _tmp = Path(DIR_DATOS) / '_apoyo_tmp'
    _tmp.mkdir(exist_ok=True)
    with zipfile.ZipFile(_apoyo_local, 'r') as _z:
        print(f'     Contenido del ZIP: {_z.namelist()}')
        _z.extractall(_tmp)

    for _nombre in _faltantes:
        _dest    = Path(DIR_DATOS) / _nombre
        _hallados = sorted(_tmp.glob(f'**/{_nombre}'))
        if _hallados:
            shutil.copy2(_hallados[0], _dest)
            print(f'  ✓  {_nombre}  ({_dest.stat().st_size/1e6:.1f} MB)')
        else:
            _todos = sorted(_tmp.glob('**/*'))
            raise RuntimeError(
                f'\n"{_nombre}" no encontrado en {_APOYO}.\n'
                f'  Archivos en el ZIP:\n'
                + '\n'.join(f'    {f.relative_to(_tmp)} ({f.stat().st_size/1e3:.0f} KB)'
                            for f in _todos if f.is_file())
            )
else:
    print('  ↩  Maestros ya disponibles')

# ── 5. Verificación final ─────────────────────────────────────────
print('\n─── Verificación ───────────────────────────────────')
_ok = True
for _f in sorted(Path(DIR_DATOS).glob(f'{MES_MMAAAA}*.csv.gz')):
    print(f'  ✓  {_f.name}  ({_f.stat().st_size/1e6:.0f} MB)')
if len(list(Path(DIR_DATOS).glob(f'{MES_MMAAAA}*.csv.gz'))) < 2:
    _ok = False
    print(f'  ✗  Faltan archivos CSV.GZ para {MES_ANALIZAR}')
for _n in (NOMBRE_MAESTRO_PROD, NOMBRE_MAESTRO_SUC):
    _p = Path(DIR_DATOS) / _n
    if _p.exists() and _p.stat().st_size > 50_000:
        print(f'  ✓  {_n}  ({_p.stat().st_size/1e6:.1f} MB)')
    else:
        _ok = False
        print(f'  ✗  {_n}  — NO ENCONTRADO')
if not _ok:
    raise RuntimeError('Faltan archivos. Revisá los errores arriba.')
print('────────────────────────────────────────────────────')
print(f'\n✓ PASO 3 completo — todo listo para procesar {MES_ANALIZAR}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 4 — Cargar y limpiar datos SEPA
# ══════════════════════════════════════════════════════════════════
import gc
import pandas as pd
from pathlib import Path

from sepa.pipeline.loader import iter_semester_csvgz, load_master_branches
from sepa.pipeline.cleaner import clean_prices
from sepa.pipeline.enricher import enrich_with_branches, filter_excluded_chains
from sepa.config.canasta import CANASTA_EANS

print(f'Cargando precios de {MES_ANALIZAR}...')
frames = []
for chunk in iter_semester_csvgz(Path(DIR_DATOS), ean_filter=CANASTA_EANS):
    chunk = filter_excluded_chains(chunk)
    chunk = clean_prices(chunk, auto_scale=True)
    frames.append(chunk)
    gc.collect()

if not frames:
    raise RuntimeError('Sin datos. Verificá que los CSV.GZ se descargaron correctamente.')

df_long = pd.concat(frames, ignore_index=True)
del frames; gc.collect()

print(f'✓ {len(df_long):,} registros | {df_long["ean_norm"].nunique()} EANs de canasta')
print(f'  Período: {df_long["fecha"].min().date()} → {df_long["fecha"].max().date()}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 5 — Enriquecer con maestros
# ══════════════════════════════════════════════════════════════════
mb = load_master_branches(f'{DIR_DATOS}/{NOMBRE_MAESTRO_SUC}')
df_enriched = enrich_with_branches(df_long, mb)
del df_long; gc.collect()

branch_cols = [c for c in ['id_comercio','id_bandera','id_sucursal'] if c in df_enriched.columns]
print(f'✓ {len(df_enriched[branch_cols].drop_duplicates()):,} sucursales | '
      f'{df_enriched.get("provincia", pd.Series(dtype=str)).nunique()} provincias | '
      f'{df_enriched.get("cadena", pd.Series(dtype=str)).nunique()} cadenas')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 6 — Canasta mensual por sucursal
# ══════════════════════════════════════════════════════════════════
from sepa.pipeline.aggregator import compute_monthly_avg, build_branch_basket

df_monthly = compute_monthly_avg(df_enriched)
df_branch  = build_branch_basket(df_monthly)
del df_monthly; gc.collect()   # ya no se necesita

df_mes = df_branch[df_branch['mes'] == MES_ANALIZAR].copy()
del df_branch; gc.collect()    # ya no se necesita

if df_mes.empty:
    raise RuntimeError(f'Sin datos para {MES_ANALIZAR}. Revisá que el ZIP contiene ese mes.')

avg = df_mes['canasta_total'].mean()
cv  = df_mes['canasta_total'].std() / avg * 100
print(f'══ ICM-UADE — {MES_ANALIZAR} ══')
print(f'Sucursales:       {len(df_mes):>8,}')
print(f'Canasta promedio: ${avg:>12,.0f}'.replace(',','.'))
print(f'Canasta mediana:  ${df_mes["canasta_total"].median():>12,.0f}'.replace(',','.'))
print(f'Dispersión (CV):  {cv:>11.1f}%')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 7 — Mapa interactivo + liberar memoria de df_enriched
# ══════════════════════════════════════════════════════════════════
from sepa.viz.maps import make_branch_map
from IPython.display import IFrame

branch_cols = [c for c in ['id_comercio','id_bandera','id_sucursal'] if c in df_enriched.columns]
suc_cols = branch_cols + [c for c in ['lat','lng','cadena','provincia','region',
                                       'sucursales_nombre','localidad_nombre','barrio_caba']
                           if c in df_enriched.columns]
df_suc = df_enriched[suc_cols].drop_duplicates(subset=branch_cols)

# Liberar df_enriched — ya no se usa (toda la info necesaria está en df_suc)
del df_enriched; gc.collect()

mapa_path = f'/content/mapa_canasta_pais_{MES_MMAAAA}_filtros.html'
make_branch_map(df_mes, df_suc, mapa_path, mes=MES_ANALIZAR)
print(f'✓ Mapa generado ({Path(mapa_path).stat().st_size/1e6:.0f} MB)')
IFrame(mapa_path, width='100%', height=600)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 8 — Rankings de cadenas
# ══════════════════════════════════════════════════════════════════
from sepa.analysis.chains import national_ranking, amba_ranking
from sepa.viz.charts import plot_chain_ranking
from IPython.display import Image, display as ipy_display

rank_nac  = national_ranking(df_mes, df_suc, mes=MES_ANALIZAR)
rank_amba = amba_ranking(df_mes, df_suc, mes=MES_ANALIZAR)

print('── Ranking Nacional ──')
display(rank_nac[['ranking','cadena','canasta_promedio','n_sucursales','vs_promedio_pct']]
        .style.format({'canasta_promedio':'${:,.0f}','vs_promedio_pct':'{:+.1f}%'}).hide(axis='index'))

png_nac = f'/content/ranking_cadenas_nacional_{MES_MMAAAA}.png'
plot_chain_ranking(rank_nac, png_nac, title=f'Ranking Nacional — {MES_ANALIZAR}')
ipy_display(Image(png_nac))

if not rank_amba.empty:
    print('\n── Ranking AMBA ──')
    display(rank_amba[['ranking','cadena','canasta_promedio','n_sucursales','vs_promedio_pct']]
            .style.format({'canasta_promedio':'${:,.0f}','vs_promedio_pct':'{:+.1f}%'}).hide(axis='index'))
    png_amba = f'/content/ranking_cadenas_amba_{MES_MMAAAA}.png'
    plot_chain_ranking(rank_amba, png_amba, title=f'Ranking AMBA — {MES_ANALIZAR}')
    ipy_display(Image(png_amba))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 9 — Ranking provincial
# ══════════════════════════════════════════════════════════════════
from sepa.viz.charts import plot_province_ranking
from IPython.display import Image, display as ipy_display

if 'provincia' in df_suc.columns:
    # Join canasta por sucursal con info provincial desde df_suc
    _br_cols = [c for c in ['id_comercio','id_bandera','id_sucursal'] if c in df_mes.columns]
    _prov_info = (df_suc[[c for c in _br_cols + ['provincia','region'] if c in df_suc.columns]]
                  .drop_duplicates(subset=_br_cols))
    _joined = df_mes.merge(_prov_info, on=_br_cols, how='left')

    _n_sin_prov = _joined['provincia'].isna().sum()
    if _n_sin_prov > 0:
        print(f'  ⚠ {_n_sin_prov} sucursales sin provincia asignada (se ignoran)')

    df_prov = (
        _joined.dropna(subset=['provincia'])
        .groupby('provincia', observed=False)['canasta_total']
        .mean()
        .reset_index()
        .rename(columns={'canasta_total': 'canasta_provincia'})
    )
    df_prov['mes'] = MES_ANALIZAR
    _avg = df_prov['canasta_provincia'].mean()
    df_prov['vs_promedio_pct'] = (df_prov['canasta_provincia'] / _avg - 1) * 100
    df_prov = df_prov.sort_values('canasta_provincia').reset_index(drop=True)
    df_prov['ranking'] = range(1, len(df_prov) + 1)

    print(f'── Ranking Provincial ({len(df_prov)} provincias) — {MES_ANALIZAR} ──')
    display(df_prov[['ranking','provincia','canasta_provincia','vs_promedio_pct']]
            .style.format({'canasta_provincia':'${:,.0f}','vs_promedio_pct':'{:+.1f}%'})
            .hide(axis='index'))

    png_prov = f'/content/ranking_provincias_{MES_MMAAAA}.png'
    plot_province_ranking(df_prov, png_prov, mes=MES_ANALIZAR)
    ipy_display(Image(png_prov))
else:
    print('⚠ Sin columna provincia en df_suc — ranking provincial no disponible')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 10 — Ranking barrios CABA
# ══════════════════════════════════════════════════════════════════
if 'barrio_caba' in df_suc.columns and 'provincia' in df_suc.columns:
    _br_cols = [c for c in ['id_comercio','id_bandera','id_sucursal'] if c in df_mes.columns]
    _caba_info = (df_suc[df_suc['provincia'] == 'CABA']
                  [[c for c in _br_cols + ['barrio_caba'] if c in df_suc.columns]]
                  .dropna(subset=['barrio_caba'])
                  .drop_duplicates(subset=_br_cols))

    _joined_caba = df_mes.merge(_caba_info, on=_br_cols, how='inner')

    if _joined_caba.empty:
        print('⚠ Sin sucursales de CABA con barrio asignado')
    else:
        df_barrios = (
            _joined_caba.groupby('barrio_caba', observed=False)['canasta_total']
            .agg(canasta_barrio='mean', n_sucursales='count')
            .reset_index()
        )
        df_barrios = df_barrios[df_barrios['n_sucursales'] >= 2]  # mínimo 2 sucursales

        _avg_caba = df_barrios['canasta_barrio'].mean()
        df_barrios['vs_promedio_caba_pct'] = (df_barrios['canasta_barrio'] / _avg_caba - 1) * 100
        df_barrios = df_barrios.sort_values('canasta_barrio', ascending=False).reset_index(drop=True)
        df_barrios['ranking'] = range(1, len(df_barrios) + 1)

        print(f'── Ranking Barrios CABA ({len(df_barrios)} barrios) — {MES_ANALIZAR} ──')
        display(df_barrios[['ranking','barrio_caba','canasta_barrio','n_sucursales','vs_promedio_caba_pct']]
                .rename(columns={'barrio_caba':'barrio','canasta_barrio':'canasta','vs_promedio_caba_pct':'vs CABA %'})
                .style.format({'canasta':'${:,.0f}','vs CABA %':'{:+.1f}%'})
                .hide(axis='index'))
else:
    print('⚠ Sin columnas barrio_caba/provincia — ranking CABA no disponible')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PASO 11 — Descargar outputs a tu computadora
# ══════════════════════════════════════════════════════════════════
from google.colab import files

outputs = [
    (f'/content/mapa_canasta_pais_{MES_MMAAAA}_filtros.html', 'mapa HTML interactivo'),
    (f'/content/ranking_cadenas_nacional_{MES_MMAAAA}.png',   'ranking nacional PNG'),
    (f'/content/ranking_cadenas_amba_{MES_MMAAAA}.png',       'ranking AMBA PNG'),
    (f'/content/ranking_provincias_{MES_MMAAAA}.png',          'ranking provincial PNG'),
]

print(f'Descargando outputs de {MES_ANALIZAR} a tu computadora...')
for ruta, label in outputs:
    if Path(ruta).exists():
        files.download(ruta)
        print(f'  ✓ {label}')
    else:
        print(f'  ⚠ No generado: {label}')

print(f'\n✓ Análisis de {MES_ANALIZAR} completado.')